# Bulls & Cows — Accuracy Evaluation

**Part A** — Base + SFT (run now)  
**Part B** — RL (run after training finishes)

---
# Part A: Environment, Model, Base + SFT Evaluation

## 1. Environment Setup

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_HOME"] = os.path.expanduser("~/venvs/gpu")
os.environ["CPATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/include"),
    os.path.expanduser("~/venvs/gpu/targets/x86_64-linux/include"),
])
os.environ["LIBRARY_PATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/lib"),
    os.path.expanduser("~/venvs/gpu/lib/stubs"),
])

## 2. Load Data

In [3]:
import json

DIFFICULTIES = [2, 3, 5, 6, 7]

all_tasks = {}
for d in DIFFICULTIES:
    path = f"data/difficulty_{d}.jsonl"
    tasks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                tasks.append(json.loads(line))
    all_tasks[d] = tasks
    print(f"Difficulty {d}: {len(tasks)} tasks (sample answer: {tasks[0]['answer']})")

Difficulty 2: 1000 tasks (sample answer: 680)
Difficulty 3: 1000 tasks (sample answer: 4672)
Difficulty 5: 1000 tasks (sample answer: 4876)
Difficulty 6: 1000 tasks (sample answer: 10857)
Difficulty 7: 1000 tasks (sample answer: 59483)


## 3. Load Model (vLLM + LoRA)

In [4]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-4B"
SFT_LORA_PATH = "outputs/sft_warmup/checkpoint-100"
MAX_SEQ_LENGTH = 6192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    fast_inference=True,
    gpu_memory_utilization=0.95,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-25 17:54:27 [vllm_utils.py:723] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.15.1.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.811 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-4B with actual GPU utilization = 90.38%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 139.81 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 6192. Num Sequences = 256.
Unsloth: vLLM's KV Cache can use up to 119.14 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.

/home/a.anokhin/venvs/gpu/lib/python3.11/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 02-25 17:54:37 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-25 17:54:37 [model.py:1561] Using max model len 6192
INFO 02-25 17:54:37 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-25 17:54:37 [vllm.py:624] Asynchronous scheduling is enabled.
INFO 02-25 17:54:38 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='unsloth/Qwen3-4B', speculative_config=None, tokenizer='unsloth/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=6192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=Fal

/home/a.anokhin/venvs/gpu/lib/python3.11/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


...

INFO 02-25 17:55:07 [gpu_model_runner.py:5063] Graph capturing finished in 10 secs, took 1.07 GiB
INFO 02-25 17:55:07 [vllm_utils.py:735] Unsloth: Patched vLLM v1 graph capture finished in 10 secs.


INFO 02-25 17:55:08 [core.py:272] init engine (profile, create kv cache, warmup model) took 24.82 seconds
INFO 02-25 17:56:02 [llm.py:343] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['q_norm', 'post_feedforward_layernorm', 'layer_norm2', 'layer_norm1', 'ffn_norm', 'norm', 'norm1', 'post_attention_layernorm', 'attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'norm2', 'input_layernorm', 'post_layernorm']


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Qwen3ForCausalLM were not initialized from the model checkpoint at unsloth/Qwen3-4B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['q_norm', 'post_feedforward_layernorm', 'layer_norm2', 'layer_norm1', 'ffn_norm', 'norm', 'norm1', 'post_attention_layernorm', 'attention_norm', 'k_norm', 'pre_feedforward_layernorm', 'norm2', 'input_layernorm', 'cross_attn_input_layernorm', 'cross_attn_post_attention_layernorm', 'post_layernorm']


## 4. Helpers

In [5]:
import re
from vllm import SamplingParams
from vllm.lora.request import LoRARequest

SYSTEM_PROMPT = """Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>"""

sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=6000,
)


def extract_answer(response: str) -> str:
    """Extract the answer from model response."""
    if "<answer>" in response and "</answer>" in response:
        ans_block = response.split("<answer>")[-1].split("</answer>")[0].strip()
        digits = re.findall(r"\d+", ans_block)
        if digits:
            return digits[-1]
    boxed = re.findall(r"\\boxed\{(\d+)\}", response)
    if boxed:
        return boxed[-1]
    digits = re.findall(r"\d{3,4}", response)
    if digits:
        return digits[-1]
    return ""


def build_prompts(tasks, tokenizer):
    """Build chat-formatted prompts for all tasks."""
    prompts = []
    for task in tasks:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task["question"]},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(text)
    return prompts


def evaluate(model, prompts, tasks, sampling_params, lora_request=None):
    """Run inference and compute accuracy."""
    outputs = model.fast_generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=lora_request,
    )
    results = []
    correct = 0
    for task, output in zip(tasks, outputs):
        response = output.outputs[0].text
        extracted = extract_answer(response)
        gold = task["answer"]
        is_correct = extracted == gold
        if is_correct:
            correct += 1
        results.append({
            "gold": gold,
            "extracted": extracted,
            "correct": is_correct,
            "response": response,
        })
    accuracy = correct / len(tasks) * 100
    return results, accuracy


print("Helpers ready.")

Helpers ready.


## 5. Evaluate Base + SFT

In [ ]:
sft_lora_request = LoRARequest(
    lora_name="sft_warmup_100",
    lora_int_id=1,
    lora_path=os.path.abspath(SFT_LORA_PATH),
)

results_map = {}  # results_map[difficulty][model_name] = (results, accuracy)

PART_A_MODELS = [
    ("Base Qwen3-4B",         None),
    ("SFT LoRA (ckpt-100)",   sft_lora_request),
]

for d in DIFFICULTIES:
    tasks = all_tasks[d]
    prompts = build_prompts(tasks, tokenizer)
    results_map[d] = {}

    print(f"\n{'='*60}")
    print(f"  DIFFICULTY {d} ({len(tasks)} tasks)")
    print(f"{'='*60}")

    for model_name, lora_req in PART_A_MODELS:
        results, accuracy = evaluate(model, prompts, tasks, sampling_params, lora_request=lora_req)
        results_map[d][model_name] = (results, accuracy)
        correct = sum(r["correct"] for r in results)
        print(f"  {model_name:<30} {correct:>3}/{len(tasks)}  = {accuracy:.1f}%")


  DIFFICULTY 2 (1000 tasks)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 

## 6. Save Base + SFT Metrics

In [6]:
import pickle

METRICS_PATH = "data/eval_base_sft.pkl"

save_data = {
    "results_map": results_map,
    "difficulties": DIFFICULTIES,
    "model_names": [name for name, _ in PART_A_MODELS],
}

with open(METRICS_PATH, "wb") as f:
    pickle.dump(save_data, f)

print(f"Base + SFT metrics saved to {METRICS_PATH}")

Base + SFT metrics saved to data/eval_base_sft.pkl


---
# Part B: RL Evaluation (run after training finishes)

Cells below load saved Base+SFT metrics, evaluate RL, and produce final comparison.

## 7. Load Saved Base + SFT Metrics

In [6]:
import pickle, json, os, re

METRICS_PATH = "data/eval_base_sft.pkl"

with open(METRICS_PATH, "rb") as f:
    saved = pickle.load(f)

results_map = saved["results_map"]
DIFFICULTIES = saved["difficulties"]

print(f"Loaded metrics for difficulties {DIFFICULTIES}")
for d in DIFFICULTIES:
    for name in saved["model_names"]:
        _, acc = results_map[d][name]
        print(f"  D{d} {name}: {acc:.1f}%")

Loaded metrics for difficulties [2, 3, 5, 6, 7]
  D2 Base Qwen3-4B: 55.3%
  D2 SFT LoRA (ckpt-100): 65.1%
  D3 Base Qwen3-4B: 8.5%
  D3 SFT LoRA (ckpt-100): 19.7%
  D5 Base Qwen3-4B: 16.4%
  D5 SFT LoRA (ckpt-100): 26.1%
  D6 Base Qwen3-4B: 0.6%
  D6 SFT LoRA (ckpt-100): 1.7%
  D7 Base Qwen3-4B: 0.3%
  D7 SFT LoRA (ckpt-100): 1.8%


## 8. Load Data + Model for RL

In [10]:
# Load tasks if not already loaded
all_tasks = {}
for d in DIFFICULTIES:
    path = f"data/difficulty_{d}.jsonl"
    tasks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                tasks.append(json.loads(line))
    all_tasks[d] = tasks

In [11]:
# Skip if model is already loaded from Part A
if 'model' not in dir():
    os.environ["CUDA_VISIBLE_DEVICES"] = "4"
    os.environ["CUDA_HOME"] = os.path.expanduser("~/venvs/gpu")
    os.environ["CPATH"] = ":".join([
        os.path.expanduser("~/venvs/gpu/include"),
        os.path.expanduser("~/venvs/gpu/targets/x86_64-linux/include"),
    ])
    os.environ["LIBRARY_PATH"] = ":".join([
        os.path.expanduser("~/venvs/gpu/lib"),
        os.path.expanduser("~/venvs/gpu/lib/stubs"),
    ])

    from unsloth import FastLanguageModel
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-4B",
        max_seq_length=6192,
        load_in_4bit=False,
        fast_inference=True,
        gpu_memory_utilization=0.95,
    )
    print("Model loaded.")
else:
    print("Model already in memory, reusing.")

Model already in memory, reusing.


In [12]:
# Re-define helpers if needed (e.g. running Part B in fresh kernel)
from vllm import SamplingParams
from vllm.lora.request import LoRARequest

SYSTEM_PROMPT = """Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>"""

sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    max_tokens=6000,
)


def extract_answer(response: str) -> str:
    if "<answer>" in response and "</answer>" in response:
        ans_block = response.split("<answer>")[-1].split("</answer>")[0].strip()
        digits = re.findall(r"\d+", ans_block)
        if digits:
            return digits[-1]
    boxed = re.findall(r"\\boxed\{(\d+)\}", response)
    if boxed:
        return boxed[-1]
    digits = re.findall(r"\d{3,4}", response)
    if digits:
        return digits[-1]
    return ""


def build_prompts(tasks, tokenizer):
    prompts = []
    for task in tasks:
        text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task["question"]},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(text)
    return prompts


def evaluate(model, prompts, tasks, sampling_params, lora_request=None):
    outputs = model.fast_generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=lora_request,
    )
    results = []
    correct = 0
    for task, output in zip(tasks, outputs):
        response = output.outputs[0].text
        extracted = extract_answer(response)
        gold = task["answer"]
        is_correct = extracted == gold
        if is_correct:
            correct += 1
        results.append({
            "gold": gold,
            "extracted": extracted,
            "correct": is_correct,
            "response": response,
        })
    accuracy = correct / len(tasks) * 100
    return results, accuracy

## 9. Evaluate RL Model

In [ ]:
RL_LORA_PATH = "/home/a.anokhin/buls_and_cow/outputs/rl_after_sft_2/checkpoint-600"  # <-- set checkpoint

rl_lora_request = LoRARequest(
    lora_name="rl_after_sft_2",
    lora_int_id=2,
    lora_path=os.path.abspath(RL_LORA_PATH),
)

RL_MODEL_NAME = "RL LoRA"  # display name for the summary table

for d in DIFFICULTIES:
    tasks = all_tasks[d]
    prompts = build_prompts(tasks, tokenizer)

    print(f"\n{'='*60}")
    print(f"  DIFFICULTY {d} ({len(tasks)} tasks)")
    print(f"{'='*60}")

    results, accuracy = evaluate(model, prompts, tasks, sampling_params, lora_request=rl_lora_request)
    results_map[d][RL_MODEL_NAME] = (results, accuracy)
    correct = sum(r["correct"] for r in results)
    print(f"  {RL_MODEL_NAME:<30} {correct:>3}/{len(tasks)}  = {accuracy:.1f}%")


  DIFFICULTY 2 (1000 tasks)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 

---
# Final Comparison (Base + SFT + RL)

## 10. Summary Table

In [14]:
# Collect all model names present in results_map
all_model_names = list(results_map[DIFFICULTIES[0]].keys())

print("=" * 70)
print("  ACCURACY COMPARISON ACROSS DIFFICULTIES")
print("=" * 70)

header = f"{'Model':<30}"
for d in DIFFICULTIES:
    header += f" {'D' + str(d):>8}"
header += f" {'Avg':>8}"
print(header)
print("-" * 70)

for model_name in all_model_names:
    row = f"{model_name:<30}"
    accs = []
    for d in DIFFICULTIES:
        _, accuracy = results_map[d][model_name]
        row += f" {accuracy:>7.1f}%"
        accs.append(accuracy)
    avg = sum(accs) / len(accs)
    row += f" {avg:>7.1f}%"
    print(row)

print("-" * 70)

# Deltas vs base
base_name = all_model_names[0]
for model_name in all_model_names[1:]:
    row = f"{'Δ ' + model_name + ' vs Base':<30}"
    deltas = []
    for d in DIFFICULTIES:
        _, base_acc = results_map[d][base_name]
        _, model_acc = results_map[d][model_name]
        diff = model_acc - base_acc
        row += f" {'+' if diff >= 0 else ''}{diff:>6.1f}%"
        deltas.append(diff)
    avg_d = sum(deltas) / len(deltas)
    row += f" {'+' if avg_d >= 0 else ''}{avg_d:>6.1f}%"
    print(row)

print("=" * 70)

  ACCURACY COMPARISON ACROSS DIFFICULTIES
Model                                D2       D3       D5       D6       D7      Avg
----------------------------------------------------------------------
Base Qwen3-4B                     55.3%     8.5%    16.4%     0.6%     0.3%    16.2%
SFT LoRA (ckpt-100)               65.1%    19.7%    26.1%     1.7%     1.8%    22.9%
RL LoRA                           91.1%    44.1%    52.3%     5.9%     7.3%    40.1%
----------------------------------------------------------------------
Δ SFT LoRA (ckpt-100) vs Base  +   9.8% +  11.2% +   9.7% +   1.1% +   1.5% +   6.7%
Δ RL LoRA vs Base              +  35.8% +  35.6% +  35.9% +   5.3% +   7.0% +  23.9%


## 11. Sample Outputs

In [ ]:
all_model_names = list(results_map[DIFFICULTIES[0]].keys())

for d in DIFFICULTIES:
    print(f"\n{'='*60}")
    print(f"  DIFFICULTY {d} — First 10 Results")
    print(f"{'='*60}")

    for model_name in all_model_names:
        results, accuracy = results_map[d][model_name]
        print(f"\n  [{model_name}] — {accuracy:.1f}%")
        for i, r in enumerate(results[:10]):
            status = "✓" if r["correct"] else "✗"
            print(f"    [{status}] #{i+1}: Gold={r['gold']}  Extracted={r['extracted']}")

## 12. Per-Task Breakdown

In [ ]:
all_model_names = list(results_map[DIFFICULTIES[0]].keys())

for d in DIFFICULTIES:
    all_res = {name: results_map[d][name][0] for name in all_model_names}
    n = len(all_res[all_model_names[0]])

    all_correct = sum(1 for i in range(n) if all(all_res[m][i]["correct"] for m in all_model_names))
    all_wrong = sum(1 for i in range(n) if not any(all_res[m][i]["correct"] for m in all_model_names))

    print(f"\nDifficulty {d}:")
    print(f"  All {len(all_model_names)} correct:  {all_correct:>5}")
    print(f"  All {len(all_model_names)} wrong:    {all_wrong:>5}")

    for idx in range(1, len(all_model_names)):
        prev = all_model_names[idx - 1]
        curr = all_model_names[idx]
        fixed = sum(1 for i in range(n) if not all_res[prev][i]["correct"] and all_res[curr][i]["correct"])
        broke = sum(1 for i in range(n) if all_res[prev][i]["correct"] and not all_res[curr][i]["correct"])
        print(f"  {curr} fixed vs {prev}: {fixed:>3}")
        print(f"  {curr} broke vs {prev}: {broke:>3}")

## 13. Save All Results

In [ ]:
all_model_names = list(results_map[DIFFICULTIES[0]].keys())
output_all = {}

for d in DIFFICULTIES:
    tasks = all_tasks[d]
    entry = {"num_tasks": len(tasks)}

    for model_name in all_model_names:
        _, accuracy = results_map[d][model_name]
        key = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
        entry[f"{key}_accuracy"] = accuracy

    per_task = []
    for i in range(len(tasks)):
        item = {"gold": tasks[i]["answer"]}
        for model_name in all_model_names:
            results = results_map[d][model_name][0]
            key = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
            item[f"{key}_extracted"] = results[i]["extracted"]
            item[f"{key}_correct"] = results[i]["correct"]
        per_task.append(item)

    entry["per_task"] = per_task
    output_all[f"difficulty_{d}"] = entry

with open("data/eval_results_all_difficulties.json", "w") as f:
    json.dump(output_all, f, indent=2)

print("Results saved to data/eval_results_all_difficulties.json")

In [7]:
import pickle

METRICS_PATH = "data/eval_base_sft.pkl"

with open(METRICS_PATH, "rb") as f:
    saved = pickle.load(f)

results_map = saved["results_map"]
DIFFICULTIES = saved["difficulties"]
model_names = saved["model_names"]

print(f"Difficulties: {DIFFICULTIES}")
print(f"Models: {model_names}")


Difficulties: [2, 3, 5, 6, 7]
Models: ['Base Qwen3-4B', 'SFT LoRA (ckpt-100)']


In [8]:
print("=" * 70)
print("  ACCURACY COMPARISON")
print("=" * 70)

header = f"{'Model':<30}"
for d in DIFFICULTIES:
    header += f" {'D' + str(d):>8}"
header += f" {'Avg':>8}"
print(header)
print("-" * 70)

for name in model_names:
    row = f"{name:<30}"
    accs = []
    for d in DIFFICULTIES:
        _, acc = results_map[d][name]
        row += f" {acc:>7.1f}%"
        accs.append(acc)
    avg = sum(accs) / len(accs)
    row += f" {avg:>7.1f}%"
    print(row)

print("-" * 70)

# Delta vs base
base = model_names[0]
for name in model_names[1:]:
    row = f"{'Δ ' + name + ' vs Base':<30}"
    deltas = []
    for d in DIFFICULTIES:
        diff = results_map[d][name][1] - results_map[d][base][1]
        row += f" {'+' if diff >= 0 else ''}{diff:>6.1f}%"
        deltas.append(diff)
    avg_d = sum(deltas) / len(deltas)
    row += f" {'+' if avg_d >= 0 else ''}{avg_d:>6.1f}%"
    print(row)

print("=" * 70)


  ACCURACY COMPARISON
Model                                D2       D3       D5       D6       D7      Avg
----------------------------------------------------------------------
Base Qwen3-4B                     55.3%     8.5%    16.4%     0.6%     0.3%    16.2%
SFT LoRA (ckpt-100)               65.1%    19.7%    26.1%     1.7%     1.8%    22.9%
----------------------------------------------------------------------
Δ SFT LoRA (ckpt-100) vs Base  +   9.8% +  11.2% +   9.7% +   1.1% +   1.5% +   6.7%


In [ ]:
RL_ONLY_LORA_PATH = "outputs/rl_after_sft_3/checkpoint-600"

rl_only_lora_request = LoRARequest(
    lora_name="rl_no_sft_600",
    lora_int_id=3,
    lora_path=os.path.abspath(RL_ONLY_LORA_PATH),
)

RL_ONLY_NAME = "RL-only (ckpt-600)"

for d in [3, 5, 6, 7]:
    tasks = all_tasks[d]
    prompts = build_prompts(tasks, tokenizer)

    print(f"\n{'='*60}")
    print(f"  DIFFICULTY {d} ({len(tasks)} tasks)")
    print(f"{'='*60}")

    results, accuracy = evaluate(model, prompts, tasks, sampling_params, lora_request=rl_only_lora_request)
    results_map[d][RL_ONLY_NAME] = (results, accuracy)
    correct = sum(r["correct"] for r in results)
    print(f"  {RL_ONLY_NAME:<30} {correct:>3}/{len(tasks)}  = {accuracy:.1f}%")



  DIFFICULTY 3 (1000 tasks)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

WARNING 02-25 17:56:23 [input_processor.py:287] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|  | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  RL-only (ckpt-600)              72/1000  = 7.2%

  DIFFICULTY 5 (1000 tasks)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%|  | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  RL-only (ckpt-600)             129/1000  = 12.9%

  DIFFICULTY 6 (1000 tasks)


Adding requests:   0%|          | 0/1000 [00:00<?, ?it/s]

Processed prompts:   0%|  | 0/1000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
import pickle

save_data = {
    "results_map": results_map,
    "difficulties": DIFFICULTIES,
    "model_names": list(results_map[DIFFICULTIES[0]].keys()),
}

with open("data/eval_base_sft.pkl", "wb") as f:
    pickle.dump(save_data, f)

print(f"Updated pkl with models: {save_data['model_names']}")


Updated pkl with models: ['Base Qwen3-4B', 'SFT LoRA (ckpt-100)']
